Roll Number: 24F-AI-120
Section: A3

In [17]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score

# Load the dataset (Make sure train.csv is in your folder)
df = pd.read_csv('train.csv')

# Keep it simple: Select only a few important features
# GrLivArea = Area, BedroomAbvGr = Rooms, Neighborhood = Location
features = ['GrLivArea', 'BedroomAbvGr', 'Neighborhood', 'SalePrice']
df = df[features].copy()

# Drop any missing values to keep it easy
df = df.dropna()

In [23]:
#Step 2: Preprocessing

# Outlier Removal
Q1 = df['GrLivArea'].quantile(0.25)
Q3 = df['GrLivArea'].quantile(0.75)
IQR = Q3 - Q1
# Keep only rows within the normal range
df = df[~((df['GrLivArea'] < (Q1 - 1.5 * IQR)) | (df['GrLivArea'] > (Q3 + 1.5 * IQR)))]

# 2. Create Categories (Low, Medium, High) based on SalePrice
df['PriceCategory'] = pd.qcut(df['SalePrice'], q=3, labels=['Low', 'Medium', 'High'])

# 3. Separate features (X) and targets (y) BEFORE encoding!
X = df.drop(['SalePrice', 'PriceCategory'], axis=1)
y_reg = df['SalePrice']          # For Regression (Continuous)
y_clf = df['PriceCategory']      # For Classification (Discrete)

# 4. Now convert text columns in X into numbers
X = pd.get_dummies(X, drop_first=True)

# 5. Scale the features (makes models work better)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [25]:
# Split data: 80% for training, 20% for testing
# We use the same random_state to ensure the rows match for both tasks
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_reg, test_size=0.2, random_state=42
)

# Split again for the classification target (using same X_train/X_test)
_, _, y_train_clf, y_test_clf = train_test_split(
    X_scaled, y_clf, test_size=0.2, random_state=42
)

In [26]:
# Initialize the Linear Regression model
lin_reg = LinearRegression()

# Train the model
lin_reg.fit(X_train, y_train_reg)

# Make predictions on the test set
y_pred_reg = lin_reg.predict(X_test)

# Calculate R-squared (how well the model explains the price)
r2 = r2_score(y_test_reg, y_pred_reg)
print(f"Linear Regression R-squared: {r2:.4f}")

Linear Regression R-squared: 0.8066


In [27]:
# Initialize the Random Forest Classifier
rf_clf = RandomForestClassifier(random_state=42)

# training the model
rf_clf.fit(X_train, y_train_clf)

# Make predictions on the test set
y_pred_clf = rf_clf.predict(X_test)

# Calculate Accuracy (how often it predicts Low/Medium/High correctly)
accuracy = accuracy_score(y_test_clf, y_pred_clf)
print(f"Random Forest Accuracy: {accuracy:.4f}")

Random Forest Accuracy: 1.0000


In [28]:
# Print a simple comparison summary
print("\n--- Performance Comparison ---")
print(f"Regression (Price Prediction) R² Score: {r2:.4f}")
print(f"Classification (Category Prediction) Accuracy: {accuracy:.4f}")

# Note: R² ranges from 0 to 1 (higher is better).
# Accuracy ranges from 0 to 1 (higher is better).


--- Performance Comparison ---
Regression (Price Prediction) R² Score: 0.8066
Classification (Category Prediction) Accuracy: 1.0000
